In [24]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

In [115]:
CORPUS = set()
data = []

genre_links = [
    "https://imsdb.com/genre/Action",
    "https://imsdb.com/genre/Comedy",
    "https://imsdb.com/genre/Romance",
    "https://imsdb.com/genre/Thriller",
    "https://imsdb.com/genre/Horror",
    "https://imsdb.com/genre/Drama",
    "https://imsdb.com/genre/Fantasy",
    "https://imsdb.com/genre/Musical",
    "https://imsdb.com/genre/Sci-Fi",
    "https://imsdb.com/genre/Mystery",
    "https://imsdb.com/genre/Crime",
    "https://imsdb.com/genre/War",
    "https://imsdb.com/genre/Family"
]

not_read = 0
read = 0

for genre_link in genre_links:
    response = requests.get(genre_link)
    content = response.text

    # Check if request works
    print("Status code:", response.status_code)

    soup = BeautifulSoup(content, "html.parser")

    movie_links = [a["href"] for a in soup.select("p a")]

    for movie_link in movie_links[:100]:    
        movie = urljoin(genre_link, movie_link)
        new_response = requests.get(movie)
        new_content = new_response.text

        print("Status code:", movie, new_response.status_code)

        new_soup = BeautifulSoup(new_content, "html.parser")

        title = new_soup.title.get_text(strip=True).split(" Script")[0]

        genre = genre_link.split("/")[-1]

        for a in new_soup.select("a"):
            if "Read" in a.text:
                script_url = a["href"]
                script_site = urljoin(movie, script_url)

                new_new_response = requests.get(script_site)
                new_new_content = new_new_response.text

                print("Status code:", script_site, new_new_response.status_code)

                if new_new_response.ok:
                    new_new_soup = BeautifulSoup(new_new_content, "html.parser")

                    script = new_new_soup.find("pre")

                    # Check if pre exists
                    if script:
                        words = script.get_text().split()
                        CORPUS.update(words)

                        data.append({
                            "title": title,
                            "genre": genre,
                            "script": script.get_text("\n", strip=True)
                        })
                        read += 1
                    else:
                        not_read += 1
                else:
                    not_read += 1

print(f"Corpus size: {len(CORPUS)}, Not read: {not_read}, Read: {read}, Sucess rate: {read / (not_read + read) * 100}")
df = pd.DataFrame(data)
df10 = df.head(10)
print(df10)
print(df['genre'].value_counts())

Status code: 200
Status code: https://imsdb.com/Movie Scripts/15 Minutes Script.html 200
Status code: https://imsdb.com/scripts/15-Minutes.html 200
Status code: https://imsdb.com/Movie Scripts/2012 Script.html 200
Status code: https://imsdb.com/scripts/2012.html 200
Status code: https://imsdb.com/Movie Scripts/30 Minutes or Less Script.html 200
Status code: https://imsdb.com/scripts/30-Minutes-or-Less.html 200
Status code: https://imsdb.com/Movie Scripts/48 Hrs. Script.html 200
Status code: https://imsdb.com/scripts/48-Hrs..html 200
Status code: https://imsdb.com/Movie Scripts/A Most Violent Year Script.html 200
Status code: https://imsdb.com/scripts/A-Most-Violent-Year.html 200
Status code: https://imsdb.com/Movie Scripts/Above the Law Script.html 200
Status code: https://imsdb.com/scripts/Above-the-Law.html 200
Status code: https://imsdb.com/Movie Scripts/Abyss, The Script.html 200
Status code: https://imsdb.com/scripts/Abyss,-The.html 200
Status code: https://imsdb.com/Movie Scripts

In [136]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from", "had", "has", "have",
    "he", "her", "hers", "him", "his", "i", "if", "in", "into", "is", "it", "its", "me", "my",
    "of", "on", "or", "our", "she", "that", "the", "their", "them", "they", "this", "to", "was",
    "we", "were", "with", "you", "your", "ext", "int", "just", "like", "cont", "continued", "look",
    "man", "looks", "night", "day", "don", "door", "know", "room", "ll", "away", "oh", "got", "turns",
    "face", "hands", "eyes", "ve"
}

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    return [word for word in tokens if word not in STOP_WORDS and len(word) > 1]

# create script variable that's a string
scripts = df['script'].astype(str)

# clean the scripts
clean_scripts = scripts.apply(clean_text)

# turn cleaned list of tokens into string
all_tokens = []
for tokens in clean_scripts:
    specific_tokens = " ".join(tokens)
    all_tokens.append(specific_tokens)

tfidf = TfidfVectorizer(max_features=20, stop_words='english')
tf_times_idf = tfidf.fit_transform(all_tokens)

# make a dataframe with rows for script and columns as words
tfidf_df = pd.DataFrame(tf_times_idf.toarray(), columns=tfidf.get_feature_names_out())
tfidf_df['genre'] = df['genre'].values

genre_tfidf = tfidf_df.groupby('genre').mean()

# Show top 20 words per genre
for genre in genre_tfidf.index:
    top_words = genre_tfidf.loc[genre].sort_values(ascending=False)[:20]
    print(f"\nTop 20 TF-IDF words per genre: {genre}")
    print(top_words.round(4))




Top TF-IDF words for genre: Action
right      0.2694
hand       0.2607
head       0.2421
time       0.2285
car        0.2216
way        0.2194
cut        0.1966
open       0.1951
pulls      0.1949
come       0.1894
going      0.1864
takes      0.1827
want       0.1705
think      0.1619
let        0.1580
looking    0.1561
good       0.1475
little     0.1364
did        0.1281
house      0.1155
Name: Action, dtype: float64

Top TF-IDF words for genre: Comedy
right      0.2834
think      0.2462
going      0.2367
car        0.2348
want       0.2270
time       0.2238
good       0.2166
come       0.1901
head       0.1892
cut        0.1879
hand       0.1771
little     0.1766
did        0.1744
way        0.1727
let        0.1709
house      0.1659
takes      0.1574
looking    0.1492
pulls      0.1279
open       0.1099
Name: Comedy, dtype: float64

Top TF-IDF words for genre: Crime
car        0.3723
right      0.2572
want       0.2218
cut        0.2044
hand       0.2024
house      0.2007
time   

In [137]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X = tfidf.fit_transform(all_tokens) # features
y = df['genre'] # target

In [138]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
# 70% for training, 30% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"\nFeatures shape: {X_train.shape[1]}")

Training set size: 730
Testing set size: 313

Features shape: 97265


In [139]:
from sklearn.neighbors import KNeighborsClassifier

# Test different values of k to find the optimal one
k_values = range(1, 21)
train_accuracies_knn = []
test_accuracies_knn = []

# Train KNN models with different k values
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    train_acc = knn.score(X_train, y_train)
    test_acc = knn.score(X_test, y_test)

    train_accuracies_knn.append(train_acc)
    test_accuracies_knn.append(test_acc)

# Find optimal k
optimal_k = k_values[np.argmax(test_accuracies_knn)]
print(f"Optimal k value: {optimal_k}")
print(f"Test accuracy with k={optimal_k}: {max(test_accuracies_knn):.4f}")

Optimal k value: 17
Test accuracy with k=17: 0.1022


In [121]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Train final KNN model with optimal k
knn_model = KNeighborsClassifier(n_neighbors=optimal_k)
knn_model.fit(X_train, y_train)

# Make predictions
y_train_pred_knn = knn_model.predict(X_train)
y_test_pred_knn = knn_model.predict(X_test)

# Calculate accuracy
train_accuracy_knn = accuracy_score(y_train, y_train_pred_knn)
test_accuracy_knn = accuracy_score(y_test, y_test_pred_knn)

print("KNN RESULTS (k=" + str(optimal_k) + ")")
print("="*50)
print(f"Training Accuracy: {train_accuracy_knn:.4f} ({train_accuracy_knn*100:.2f}%)")
print(f"Testing Accuracy: {test_accuracy_knn:.4f} ({test_accuracy_knn*100:.2f}%)")

# Confusion matrix
cm_knn = confusion_matrix(y_test, y_test_pred_knn)
print("\nConfusion Matrix:")
print(cm_knn)

KNN RESULTS (k=19)
Training Accuracy: 0.2356 (23.56%)
Testing Accuracy: 0.1086 (10.86%)

Confusion Matrix:
[[ 7  1  0  7  0  8  2  0  0  2  4  0  0]
 [ 5  3  0  5  1  4  2  0  0  2  1  0  0]
 [ 4  0  2  9  2  8  3  0  1  1  1  0  0]
 [ 6  0  2  7  3  5  3  0  1  1  2  0  0]
 [ 1  1  0  2  0  8  0  0  0  0  0  0  0]
 [ 5  1  0  3  0 12  3  0  0  0  3  0  0]
 [ 2  2  2  2  0 11  0  1  3  1  1  0  0]
 [ 0  0  0  1  0  4  1  0  0  1  1  0  0]
 [ 4  0  2  2  0  8  4  1  0  3  2  0  0]
 [ 5  2  1  6  0  9  3  0  0  1  0  0  0]
 [ 4  1  0  1  0 10  1  0  0  1  2  1  0]
 [11  2  3  6  1 10  4  1  0  2  3  0  0]
 [ 1  0  1  4  0  1  1  0  0  0  1  0  0]]
